# Activity 10 — K-Means Clustering & Support Vector Machines: Digital Marketing Analytics

**Course:** Introduction to Artificial Intelligence  
**Sessions:** 29 (K-Means Algorithm) and 31 (Support Vector Machines)  
**Topic:** Unsupervised Learning · Supervised Classification · Hyperparameter Tuning

---

## Scenario: NovaPulse Media — StyleHub Campaign Analytics

You are a data scientist at **NovaPulse Media**, a digital marketing agency. Your client is **StyleHub**, a mid-size online fashion retailer with 200,000 registered customers.

StyleHub is planning its largest promotional campaign of the year — a **Spring Sale event** — and the marketing team wants to:

1. **Understand who their customers are** by segmenting them into behaviorally distinct groups, so each group can receive a tailored email campaign (instead of a generic blast).
2. **Predict which customers will click on a retargeting ad**, so the paid advertising budget is spent efficiently on high-probability converters.

These are two classic AI problems in digital marketing:
- **Task 1 — Customer Segmentation** using **K-Means Clustering** (Session 29) — unsupervised, no labels needed.
- **Task 2 — Ad Click Prediction** using **Support Vector Machines** (Session 31) — supervised, trained on historical click data.

---

## Dataset Description

### Task 1 — Customer Behavioral Data (200 customers, unlabeled)

| Feature | Description | Unit |
|---------|-------------|------|
| `session_duration` | Average minutes per browsing session | minutes |
| `pages_per_visit` | Average pages viewed per session | count |
| `purchase_freq` | Number of purchases in the last 3 months | count/month |
| `avg_order_value` | Average dollar amount per order | USD |
| `email_ctr` | Email campaign click-through rate | % |
| `recency_days` | Days since last site visit | days |

### Task 2 — Ad Click History (300 customers, labeled)

| Feature | Description | Unit |
|---------|-------------|------|
| `age` | Customer age | years |
| `income_k` | Annual household income | $k |
| `browsing_hours` | Weekly hours browsing StyleHub | hours |
| `pages_viewed` | Pages viewed in last session | count |
| `prev_purchases` | Total purchases in account history | count |
| `ad_clicked` | **Target** — clicked the retargeting ad | 0 = No / 1 = Yes |

In [ ]:
# Setup — Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, silhouette_score)
from sklearn.pipeline import Pipeline

import ipywidgets as widgets
from ipywidgets import interact

import warnings
warnings.filterwarnings('ignore')

# Confirm library versions
import sklearn
print('Library versions:')
print(f'  numpy     : {np.__version__}')
print(f'  pandas    : {pd.__version__}')
print(f'  sklearn   : {sklearn.__version__}')
print(f'  ipywidgets: {widgets.__version__}')
print()
print('All imports successful. Ready to begin.')

---

# TASK 1 — Customer Segmentation with K-Means Clustering

## Background (Session 29)

**K-Means** is an *unsupervised* learning algorithm — it finds structure in data **without labels**. The goal is to partition N customers into K groups (clusters) such that customers within a cluster are more similar to each other than to customers in other clusters.

### The K-Means Algorithm (3 steps, repeated until convergence):

1. **Initialize** K centroids — randomly selected points in the feature space.
2. **Assign** every data point to the nearest centroid (measured by Euclidean distance).
3. **Update** each centroid to the mean of all points assigned to it.

Repeat steps 2–3 until no point changes its cluster assignment (convergence).

### Why Segmentation Matters in Marketing

A one-size-fits-all email blast typically achieves 15–20% open rates. Targeted campaigns that speak to a customer's specific behavior routinely outperform generic campaigns by 30–60%. By discovering natural behavioral groups, NovaPulse can craft three distinct messages:
- High-value loyalists → VIP early access
- Budget browsers → price-led promotions
- Occasional buyers → re-engagement offers

---

## Step 1.1 — Generate Customer Dataset and Summary Statistics

In [ ]:
# ── Step 1.1 ─ Generate StyleHub customer behavioral dataset ──────────────────
#
# We simulate 200 customers drawn from three underlying behavioral populations.
# In a real engagement, these metrics would come from web analytics (e.g., Google
# Analytics, Klaviyo) and a CRM system.  The random seed is fixed so every student
# works with identical data.
#
# Three latent populations (unknown to K-Means — it must discover them):
#   Population 0 : Budget Browsers  (n=85)  — long sessions, many pages, rarely buy
#   Population 1 : Loyal High-Value (n=75)  — short efficient sessions, buy often, big orders
#   Population 2 : Bargain Hunters  (n=40)  — moderate engagement, occasional sales purchases

np.random.seed(10)

features_km = ['session_duration', 'pages_per_visit', 'purchase_freq',
               'avg_order_value', 'email_ctr', 'recency_days']

# Population 0: Budget Browsers
seg0 = np.column_stack([
    np.clip(np.random.normal(15, 3, 85),  5,  30),   # session_duration (min)
    np.clip(np.random.normal(10, 2, 85),  2,  20),   # pages_per_visit
    np.clip(np.random.normal( 0.5, 0.3, 85), 0.0, 2.0),  # purchase_freq
    np.clip(np.random.normal(35,  8, 85), 10,  70),  # avg_order_value ($)
    np.clip(np.random.normal(12,  4, 85),  2,  30),  # email_ctr (%)
    np.clip(np.random.normal( 7,  3, 85),  1,  20),  # recency_days
])

# Population 1: Loyal High-Value
seg1 = np.column_stack([
    np.clip(np.random.normal( 7,  2, 75),  2,  15),  # session_duration
    np.clip(np.random.normal( 5,  1.5, 75), 1, 10),  # pages_per_visit
    np.clip(np.random.normal( 4.0, 0.8, 75), 2, 7),  # purchase_freq
    np.clip(np.random.normal(120, 25, 75), 60, 200),  # avg_order_value ($)
    np.clip(np.random.normal(38,  6, 75), 20,  55),  # email_ctr (%)
    np.clip(np.random.normal( 3,  2, 75),  0,   8),  # recency_days
])

# Population 2: Bargain Hunters
seg2 = np.column_stack([
    np.clip(np.random.normal(10,  3, 40),  4,  20),  # session_duration
    np.clip(np.random.normal( 7,  2, 40),  2,  14),  # pages_per_visit
    np.clip(np.random.normal( 1.5, 0.5, 40), 0.5, 3.0),  # purchase_freq
    np.clip(np.random.normal(55, 15, 40), 20,  90),  # avg_order_value ($)
    np.clip(np.random.normal(22,  5, 40), 10,  38),  # email_ctr (%)
    np.clip(np.random.normal(18,  6, 40),  5,  35),  # recency_days
])

X_km = np.vstack([seg0, seg1, seg2])  # shape: (200, 6)
df_km = pd.DataFrame(X_km, columns=features_km).round(2)

N_km = len(df_km)
print(f'Dataset shape : {df_km.shape}  ({N_km} customers, {len(features_km)} features)')
print(f'Random seed   : 10')
print()
print('Summary statistics:')
print(df_km.describe().round(2).to_string())

---

## Step 1.2 — Exploratory Data Analysis

Before clustering, we need to understand the data distribution of each feature. Two things to look for:

1. **Do any features show multi-modal distributions?** A histogram with two or three humps suggests the presence of distinct underlying sub-groups — exactly what K-Means hopes to find.
2. **Are the features on very different scales?** If `avg_order_value` ranges from \$10–\$200 while `purchase_freq` ranges 0–7, the distance calculation will be dominated by order value. This is why normalization (Step 1.3) is critical.

In [ ]:
# ── Step 1.2 ─ EDA: Feature distributions and correlation ────────────────────

feat_labels = [
    'Session Duration\n(minutes)', 'Pages per Visit',
    'Purchase Freq\n(per month)',  'Avg Order Value\n(USD)',
    'Email CTR (%)',               'Recency\n(days since last visit)'
]

fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 4, figure=fig)

# Feature histograms (top 2 rows, first 3 cols each)
hist_positions = [gs[0,0], gs[0,1], gs[0,2], gs[1,0], gs[1,1], gs[1,2]]
colors = ['steelblue','seagreen','firebrick','darkorange','mediumpurple','sienna']

for feat, pos, label, color in zip(features_km, hist_positions, feat_labels, colors):
    ax = fig.add_subplot(pos)
    ax.hist(df_km[feat], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(feat, fontsize=10)

# Correlation heatmap (right column, both rows)
ax_corr = fig.add_subplot(gs[:, 3])
corr = df_km.corr()
im = ax_corr.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, interpolation='nearest')
plt.colorbar(im, ax=ax_corr, fraction=0.046, pad=0.04)
ticks = range(len(features_km))
short_labels = ['sess_dur','pages','purch_freq','order_val','email_ctr','recency']
ax_corr.set_xticks(ticks)
ax_corr.set_yticks(ticks)
ax_corr.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax_corr.set_yticklabels(short_labels, fontsize=8)
for i in range(len(features_km)):
    for j in range(len(features_km)):
        ax_corr.text(j, i, f'{corr.iloc[i,j]:.2f}',
                     ha='center', va='center', fontsize=7,
                     color='black' if abs(corr.iloc[i,j]) < 0.6 else 'white')
ax_corr.set_title('Feature Correlation\nMatrix', fontsize=10)

plt.suptitle('Step 1.2 — EDA: Feature Distributions and Correlation Matrix',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Feature ranges (before normalization):')
for feat in features_km:
    print(f'  {feat:<22} min={df_km[feat].min():7.2f}  max={df_km[feat].max():7.2f}  '
          f'mean={df_km[feat].mean():7.2f}  std={df_km[feat].std():6.2f}')

---

## Step 1.3 — Feature Normalization

K-Means computes **Euclidean distance** between every point and every centroid. If one feature spans a large numerical range, it will dominate the distance — the clusters will be defined almost entirely by that feature, and the others will contribute almost nothing.

For example:
- `avg_order_value` ranges ~\$10–\$200 (range ≈ 190)
- `purchase_freq` ranges ~0–7 (range ≈ 7)

Without normalization, a \$100 difference in order value completely swamps a 5-purchase difference in frequency. After applying `StandardScaler` (z = (x − μ) / σ), every feature has **mean = 0** and **std = 1**, so all features contribute equally to distance.

> **Important:** For unsupervised learning (no train/test split), we fit the scaler on the **entire dataset**.

In [ ]:
# ── Step 1.3 ─ Feature Normalization ─────────────────────────────────────────

scaler_km = StandardScaler()
X_km_sc   = scaler_km.fit_transform(X_km)   # fit on full dataset (no test set in clustering)

print('Feature statistics BEFORE normalization:')
for i, feat in enumerate(features_km):
    print(f'  {feat:<22} mean={X_km[:,i].mean():8.2f}  std={X_km[:,i].std():7.2f}  '
          f'range=[{X_km[:,i].min():.1f}, {X_km[:,i].max():.1f}]')

print()
print('Feature statistics AFTER normalization:')
for i, feat in enumerate(features_km):
    print(f'  {feat:<22} mean={X_km_sc[:,i].mean():8.4f}  std={X_km_sc[:,i].std():.4f}')

print()
print('All features now have mean ≈ 0 and std = 1.')
print('K-Means will treat every feature as equally important in distance calculations.')

---

## Step 1.4 — The Elbow Method: Choosing K

K-Means requires us to specify K (the number of clusters) in advance. How do we know the right K?

The **Elbow Method** plots the **inertia** (also called Within-Cluster Sum of Squares, WCSS) for K = 1, 2, …, 10:

$$\text{Inertia} = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

- **Large K** → smaller clusters → lower inertia (each point is closer to its centroid)
- **K = N** → zero inertia (each point is its own centroid) — useless

We look for the **elbow** — the value of K where inertia stops dropping sharply. This "bend" in the curve suggests that adding more clusters yields diminishing returns.

**Session 29 connection:** The elbow is a heuristic for the bias-variance tradeoff in unsupervised learning — more clusters = more flexible model = lower training error, but too many clusters overfits the noise.

In [ ]:
# ── Step 1.4 ─ Elbow Method ───────────────────────────────────────────────────

K_range  = range(1, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=10)
    km.fit(X_km_sc)
    inertias.append(km.inertia_)
    if k >= 2:
        sil_scores.append(silhouette_score(X_km_sc, km.labels_))
    else:
        sil_scores.append(None)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Elbow curve ---
axes[0].plot(K_range, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axvline(3, color='firebrick', linestyle='--', linewidth=1.5,
                label='Elbow at K = 3')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=11)
axes[0].set_title('Step 1.4 — Elbow Method: Inertia vs K', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
for k, inertia in zip(K_range, inertias):
    axes[0].text(k, inertia + 8, f'{inertia:.0f}', ha='center', fontsize=8)

# --- Silhouette scores ---
sil_k     = list(K_range)[1:]
sil_vals  = sil_scores[1:]
axes[1].bar(sil_k, sil_vals, color='seagreen', edgecolor='white', alpha=0.85)
axes[1].axvline(3, color='firebrick', linestyle='--', linewidth=1.5,
                label='K = 3 selected')
for k, s in zip(sil_k, sil_vals):
    axes[1].text(k, s + 0.005, f'{s:.3f}', ha='center', fontsize=9)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Step 1.4 — Silhouette Score vs K\n'
                  '(higher = more distinct clusters)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Elbow method results:')
print(f'{"K":>4}  {"Inertia":>10}  {"Silhouette":>12}')
print('-' * 32)
for k, inertia, sil in zip(K_range, inertias, sil_scores):
    sil_str = f'{sil:.4f}' if sil is not None else '  N/A   '
    marker  = '  <-- elbow' if k == 3 else ''
    print(f'{k:>4}  {inertia:>10.2f}  {sil_str:>12}{marker}')

---

## Step 1.5 — Fit K-Means with K = 3

Based on the elbow at K = 3, we now fit the final K-Means model. Key parameters:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `n_clusters` | 3 | Number of segments to find |
| `n_init` | 10 | Repeat with 10 random starting centroids; keep best |
| `max_iter` | 300 | Maximum iterations per run |
| `random_state` | 10 | Reproducibility |

**Why n_init = 10?** K-Means is sensitive to initialization — different starting centroids can lead to different (sub-optimal) solutions. Running it 10 times and keeping the run with lowest inertia reduces this risk significantly.

In [ ]:
# ── Step 1.5 ─ Fit K-Means (K = 3) ───────────────────────────────────────────

km3 = KMeans(n_clusters=3, n_init=10, max_iter=300, random_state=10)
km3.fit(X_km_sc)

labels_km  = km3.labels_
df_km['cluster'] = labels_km

sil3 = silhouette_score(X_km_sc, labels_km)

print('=== K-Means (K=3) Summary ===')
print(f'  Inertia (WCSS)    : {km3.inertia_:.2f}')
print(f'  Iterations run    : {km3.n_iter_}')
print(f'  Silhouette score  : {sil3:.4f}  (range -1 to +1; closer to 1 = well-separated clusters)')
print()

print('Cluster sizes:')
for c in sorted(df_km['cluster'].unique()):
    n = (df_km['cluster'] == c).sum()
    pct = n / N_km * 100
    print(f'  Cluster {c} : {n:3d} customers ({pct:.1f}%)')
print()

print('Cluster centroids (normalized feature space):')
centroid_df = pd.DataFrame(km3.cluster_centers_, columns=features_km).round(3)
print(centroid_df.to_string())

---

## Step 1.6 — Visualize Clusters with PCA (2D Projection)

Our data has 6 features — we cannot plot it directly. **Principal Component Analysis (PCA)** reduces the 6-dimensional space to 2 dimensions while preserving as much variance as possible. This gives us a visual "map" of the customer space.

> **Note:** PCA is used here *only for visualization*, not for clustering. The K-Means algorithm ran on all 6 scaled features.

In [ ]:
# ── Step 1.6 ─ PCA 2D Visualization of Clusters ──────────────────────────────

pca = PCA(n_components=2, random_state=10)
X_pca = pca.fit_transform(X_km_sc)

var_explained = pca.explained_variance_ratio_

cluster_colors  = {0: 'steelblue', 1: 'firebrick', 2: 'seagreen'}
cluster_markers = {0: 'o', 1: 's', 2: '^'}
# Tentative segment names (confirmed by Step 1.7 profile analysis)
cluster_names   = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter coloured by K-Means cluster
for c in sorted(df_km['cluster'].unique()):
    mask = df_km['cluster'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=cluster_colors[c], marker=cluster_markers[c],
                    s=60, alpha=0.75, edgecolors='white', linewidths=0.5,
                    label=f'Cluster {c}  (n={(mask).sum()})')

# Plot centroids projected into PCA space
centroids_pca = pca.transform(km3.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroids')

axes[0].set_xlabel(f'PC1  ({var_explained[0]*100:.1f}% variance)', fontsize=11)
axes[0].set_ylabel(f'PC2  ({var_explained[1]*100:.1f}% variance)', fontsize=11)
axes[0].set_title('Step 1.6 — K-Means Clusters (PCA 2D Projection)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: PCA component loadings bar chart
x = np.arange(len(features_km))
width = 0.35
axes[1].bar(x - width/2, pca.components_[0], width, label='PC1',
            color='steelblue', edgecolor='white')
axes[1].bar(x + width/2, pca.components_[1], width, label='PC2',
            color='darkorange', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['sess_dur','pages','purch_freq','order_val','email_ctr','recency'],
                         rotation=35, ha='right', fontsize=9)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('Loading (contribution to component)', fontsize=10)
axes[1].set_title(f'Step 1.6 — PCA Feature Loadings\n'
                  f'(PC1+PC2 explain {sum(var_explained)*100:.1f}% of total variance)',
                  fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'PCA variance explained:')
print(f'  PC1 : {var_explained[0]*100:.1f}%')
print(f'  PC2 : {var_explained[1]*100:.1f}%')
print(f'  Total (PC1+PC2) : {sum(var_explained)*100:.1f}%')

---

## Step 1.7 — Cluster Profile Analysis and Marketing Interpretation

The cluster numbers (0, 1, 2) are meaningless by themselves. To turn them into **actionable marketing personas**, we inspect the mean value of each feature within each cluster. This is the interpretive step that transforms a mathematical result into a business recommendation.

In [ ]:
# ── Step 1.7 ─ Cluster Profile Analysis ──────────────────────────────────────

profile = df_km.groupby('cluster')[features_km].mean().round(2)

print('=== Cluster Profiles (Mean Feature Values) ===')
print(profile.to_string())
print()

# ── Grouped bar chart of cluster profiles ────────────────────────────────────
# Normalize profiles to [0,1] per feature for visual comparison
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Raw mean values per feature per cluster ---
x = np.arange(len(features_km))
width = 0.28
colors_bar = ['steelblue', 'firebrick', 'seagreen']
for i, (c, color) in enumerate(zip(sorted(df_km['cluster'].unique()), colors_bar)):
    axes[0].bar(x + (i - 1) * width, profile.loc[c], width,
                label=f'Cluster {c}', color=color, edgecolor='white', alpha=0.9)

axes[0].set_xticks(x)
axes[0].set_xticklabels(['sess_dur\n(min)','pages\n/visit','purch_freq\n/month',
                          'order_val\n($)','email_ctr\n(%)','recency\n(days)'],
                         fontsize=9)
axes[0].set_ylabel('Mean Value (raw units)', fontsize=10)
axes[0].set_title('Step 1.7 — Cluster Profiles: Mean Feature Values', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# --- Normalized profiles (0-1 scale, spider-like bar) ---
for i, (c, color) in enumerate(zip(sorted(df_km['cluster'].unique()), colors_bar)):
    axes[1].bar(x + (i - 1) * width, profile_norm.loc[c], width,
                label=f'Cluster {c}', color=color, edgecolor='white', alpha=0.9)

axes[1].set_xticks(x)
axes[1].set_xticklabels(['sess_dur','pages','purch_freq','order_val',
                          'email_ctr','recency'], fontsize=9)
axes[1].set_ylabel('Normalized Mean (0 = lowest, 1 = highest)', fontsize=10)
axes[1].set_title('Step 1.7 — Normalized Cluster Profiles\n'
                  '(highlights relative strengths per feature)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# ── Marketing persona summary ─────────────────────────────────────────────────
print('=== NovaPulse Marketing Persona Recommendations ===')
print()

# Identify clusters by their dominant features
c_high_value = profile['avg_order_value'].idxmax()
c_browser    = profile['session_duration'].idxmax()
c_remaining  = [c for c in [0,1,2] if c not in [c_high_value, c_browser]][0]

personas = {
    c_high_value : ('Loyal High-Value Customers',
                    'VIP early access, loyalty rewards, premium product highlights'),
    c_browser    : ('Budget Browsers',
                    'Discount-led banners, "You left items in your cart" reminders'),
    c_remaining  : ('Bargain Hunters',
                    'Flash-sale alerts, "Last chance" urgency emails, bundle deals'),
}

for c, (name, strategy) in sorted(personas.items()):
    n = (df_km['cluster'] == c).sum()
    print(f'Cluster {c} → "{name}"  (n={n})')
    print(f'  Recommended strategy : {strategy}')
    print(f'  Key profile          : '
          f'avg_order=${profile.loc[c,"avg_order_value"]:.0f}, '
          f'purch_freq={profile.loc[c,"purchase_freq"]:.1f}/mo, '
          f'email_ctr={profile.loc[c,"email_ctr"]:.0f}%')
    print()

---

## Step 1.8 — Interactive Widget: Explore K and Clustering

Use the slider below to change the number of clusters K. Observe how:
- The inertia decreases as K increases
- The PCA scatter splits into more groups
- The silhouette score may peak at K = 3 (most cohesive clusters)

**Experiments to try:**
1. Set K = 2 — which two populations merge? Does the result make business sense?
2. Set K = 4 — does the new cluster reveal a meaningful sub-group, or just split an existing one?
3. Set K = 6 — is there a marketing interpretation for six segments, or is this overfitting?
4. Compare the silhouette score at K = 3 vs K = 4 — which K produces better-separated clusters?

**Reflection Question (answer in the cell below the widget):**  
At what K does the silhouette score peak in your experiment? Does the elbow in the inertia curve agree with the silhouette peak? Which criterion do you trust more and why?

In [ ]:
# ── Step 1.8 ─ Interactive K-Means Widget ─────────────────────────────────────

def run_kmeans_widget(n_clusters):
    km_w = KMeans(n_clusters=n_clusters, n_init=10, random_state=10)
    km_w.fit(X_km_sc)
    labels_w = km_w.labels_
    inertia_w = km_w.inertia_
    sil_w = silhouette_score(X_km_sc, labels_w) if n_clusters > 1 else None

    X_pca_w = PCA(n_components=2, random_state=10).fit_transform(X_km_sc)

    cmap = plt.cm.get_cmap('tab10', n_clusters)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'K = {n_clusters}  |  Inertia = {inertia_w:.1f}  |  '
        f'Silhouette = {sil_w:.4f}' if sil_w else
        f'K = {n_clusters}  |  Inertia = {inertia_w:.1f}  |  Silhouette = N/A',
        fontsize=12
    )

    # PCA scatter
    for c in range(n_clusters):
        mask = labels_w == c
        axes[0].scatter(X_pca_w[mask, 0], X_pca_w[mask, 1],
                        color=cmap(c), s=50, alpha=0.7,
                        edgecolors='white', linewidths=0.4,
                        label=f'Cluster {c} (n={mask.sum()})')
    centroids_w = PCA(n_components=2, random_state=10).fit(
        X_km_sc).transform(km_w.cluster_centers_)
    axes[0].scatter(centroids_w[:, 0], centroids_w[:, 1],
                    c='black', marker='X', s=180, zorder=5)
    axes[0].set_xlabel('PC1', fontsize=10)
    axes[0].set_ylabel('PC2', fontsize=10)
    axes[0].set_title(f'PCA 2D Projection (K={n_clusters})', fontsize=11)
    axes[0].legend(fontsize=9, loc='best')
    axes[0].grid(True, alpha=0.3)

    # Elbow curve with current K highlighted
    axes[1].plot(K_range, inertias, 'o-', color='steelblue', linewidth=2)
    axes[1].scatter([n_clusters], [inertia_w],
                    color='firebrick', s=180, zorder=5,
                    label=f'Current K={n_clusters}')
    axes[1].set_xlabel('K', fontsize=10)
    axes[1].set_ylabel('Inertia', fontsize=10)
    axes[1].set_title('Elbow Curve (current K highlighted)', fontsize=11)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Cluster size summary
    unique, counts = np.unique(labels_w, return_counts=True)
    print('Cluster sizes:')
    for c, n in zip(unique, counts):
        print(f'  Cluster {c}: {n:3d} customers ({n/N_km*100:.1f}%)')


widgets.interact(
    run_kmeans_widget,
    n_clusters=widgets.IntSlider(
        min=2, max=8, step=1, value=3,
        description='K (clusters):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%')
    )
)

**Reflection Question Q1 — answer here:**

*Write your answer in this Markdown cell.*

---

# TASK 2 — Ad Click Prediction with Support Vector Machines

## Background (Session 31)

The MRHA CardioWatch scenario (Activities 7 and 9) used regression and neural networks. Here we use **Support Vector Machines (SVMs)** — supervised classifiers that find an optimal decision boundary (hyperplane) separating two classes.

### Key SVM concepts:

- **Hyperplane**: A decision boundary of dimension (N−1) that separates class 0 from class 1.
- **Support Vectors**: The data points lying *closest* to the hyperplane. Only these points determine the boundary — all other points are irrelevant.
- **Margin**: The gap between the hyperplane and the nearest support vectors. SVMs maximize this margin → better generalization.
- **Kernel trick**: Projects data to a higher-dimensional space where a linear boundary can separate classes that are non-linearly entangled in the original space.
- **Regularization parameter C**: Controls the bias-variance tradeoff. Small C = wide margin, some misclassifications allowed. Large C = narrow margin, fewer training errors but risk of overfitting.

### Marketing application

StyleHub runs retargeting ads on social media (Meta, TikTok) and pays **per impression**. Rather than showing the ad to all 200,000 customers, NovaPulse wants to predict which customers are likely to click — so the budget is concentrated on high-probability clickers. A 10% improvement in targeting efficiency translates directly to reduced cost-per-acquisition.

---

## Step 2.1 — Generate Ad Click Dataset and EDA

In [ ]:
# ── Step 2.1 ─ Generate ad-click dataset ──────────────────────────────────────
#
# 300 customers with historical ad interaction records.
# Features from CRM + web analytics; label from ad server logs.
#
# Click probability formula (logistic model):
#   logit = -4.0 + 0.03*income_k + 0.12*pages_viewed
#           + 0.10*prev_purchases - 0.02*age + 0.05*browsing_hours
#   P(click) = sigmoid(logit)

np.random.seed(10)
N_svm = 300

age             = np.random.uniform(18, 65,  N_svm)
income_k        = np.random.uniform(20, 120, N_svm)
browsing_hours  = np.random.uniform( 0.5, 15, N_svm)
pages_viewed    = np.random.uniform( 1, 25,  N_svm)
prev_purchases  = np.random.randint( 0, 12,  N_svm).astype(float)

logit_click = (-4.0
               + 0.03 * income_k
               + 0.12 * pages_viewed
               + 0.10 * prev_purchases
               - 0.02 * age
               + 0.05 * browsing_hours)
prob_click  = 1 / (1 + np.exp(-logit_click))
ad_clicked  = (np.random.uniform(0, 1, N_svm) < prob_click).astype(int)

features_svm = ['age', 'income_k', 'browsing_hours', 'pages_viewed', 'prev_purchases']

df_svm = pd.DataFrame({
    'age'            : age.round(1),
    'income_k'       : income_k.round(1),
    'browsing_hours' : browsing_hours.round(2),
    'pages_viewed'   : pages_viewed.round(1),
    'prev_purchases' : prev_purchases,
    'ad_clicked'     : ad_clicked
})

click_counts = df_svm['ad_clicked'].value_counts().sort_index()
click_pct    = click_counts / N_svm * 100

print(f'Dataset shape : {df_svm.shape}  ({N_svm} customers, {len(features_svm)} features)')
print(f'Random seed   : 10')
print()
print('Ad click distribution:')
for cls in [0, 1]:
    label = 'Did not click (0)' if cls == 0 else 'Clicked (1)      '
    print(f'  Class {cls} — {label} : {click_counts[cls]:3d} customers ({click_pct[cls]:.1f}%)')
print()
print('Feature means by class:')
print(df_svm.groupby('ad_clicked')[features_svm].mean().round(2).to_string())

# ── EDA: class distribution bar + feature histograms ─────────────────────────
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig)

# Class distribution bar
ax_bar = fig.add_subplot(gs[:, 0])
bar_colors = ['steelblue', 'firebrick']
bars = ax_bar.bar(['No Click (0)', 'Clicked (1)'], click_counts.values,
                  color=bar_colors, edgecolor='white', width=0.5)
for bar, cnt in zip(bars, click_counts.values):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(cnt), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax_bar.set_ylabel('Number of Customers', fontsize=10)
ax_bar.set_title('Class Distribution', fontsize=11)
ax_bar.text(0.5, 0.93, f'{click_pct[1]:.1f}% clicked',
            ha='center', transform=ax_bar.transAxes, fontsize=10, color='firebrick')

# Feature histograms by class
feat_positions = [gs[0,1], gs[0,2], gs[0,3], gs[1,1], gs[1,2]]
feat_titles_svm = ['Age (years)', 'Income ($k)',
                   'Browsing Hours/week', 'Pages Viewed', 'Prev Purchases']
class_colors_svm = ['steelblue', 'firebrick']
class_labels_svm = ['No Click (0)', 'Clicked (1)']

for feat, pos, title in zip(features_svm, feat_positions, feat_titles_svm):
    ax = fig.add_subplot(pos)
    for cls, color, label in zip([0, 1], class_colors_svm, class_labels_svm):
        vals = df_svm.loc[df_svm['ad_clicked'] == cls, feat]
        ax.hist(vals, bins=16, color=color, alpha=0.65, density=True,
                edgecolor='white', label=label)
    ax.set_xlabel(title, fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Step 2.1 — EDA: Class Distribution and Feature Histograms by Class',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## Step 2.2 — Train / Test Split and Feature Normalization

SVMs are highly sensitive to feature scales for the same reason as neural networks: the optimization objective (maximizing the margin) depends on distances. We apply `StandardScaler` **fit only on the training set** to avoid data leakage into the test set.

In [ ]:
# ── Step 2.2 ─ Train / Test Split and Normalization ────────────────────────────

X_svm = df_svm[features_svm].values
y_svm = df_svm['ad_clicked'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_svm, y_svm, test_size=0.20, random_state=10, stratify=y_svm
)

# Fit scaler on training data only
scaler_svm  = StandardScaler()
X_tr_sc     = scaler_svm.fit_transform(X_tr)
X_te_sc     = scaler_svm.transform(X_te)

print('Train / Test Split:')
print(f'  Total     : {N_svm}')
print(f'  Training  : {len(y_tr)} ({len(y_tr)/N_svm*100:.0f}%)')
print(f'  Test      : {len(y_te)} ({len(y_te)/N_svm*100:.0f}%)')
print()
print('Class balance in each split:')
for split_name, split_y in [('Training', y_tr), ('Test', y_te)]:
    n1 = split_y.sum()
    n0 = len(split_y) - n1
    print(f'  {split_name:10s} : No Click={n0} ({n0/len(split_y)*100:.1f}%)  '
          f'Clicked={n1} ({n1/len(split_y)*100:.1f}%)')
print()
print('Both splits preserve the overall ~{:.0f}% click rate (stratified split).'.format(
    y_svm.mean()*100))

---

## Step 2.3 — Linear SVM

A **Linear SVM** (`kernel='linear'`) finds the hyperplane that maximally separates the two classes in the original feature space. It is appropriate when the classes are *approximately* linearly separable.

The decision function is:

$$f(x) = w \cdot x + b$$

Predict class 1 if f(x) > 0, class 0 otherwise.

The margin is $\frac{2}{\|w\|}$. Maximizing the margin is equivalent to minimizing $\|w\|$, subject to all training points being on the correct side (for hard-margin SVM). The regularization parameter **C** softens this constraint.

In [ ]:
# ── Step 2.3 ─ Linear SVM ─────────────────────────────────────────────────────

svm_lin = SVC(kernel='linear', C=1.0, random_state=10)
svm_lin.fit(X_tr_sc, y_tr)

y_pred_lin_tr = svm_lin.predict(X_tr_sc)
y_pred_lin_te = svm_lin.predict(X_te_sc)

acc_lin_tr = accuracy_score(y_tr, y_pred_lin_tr)
acc_lin_te = accuracy_score(y_te, y_pred_lin_te)

cm_lin = confusion_matrix(y_te, y_pred_lin_te)
TN, FP, FN, TP = cm_lin.ravel()

print('=== Step 2.3 — Linear SVM (C=1.0) ===')
print(f'  Train accuracy : {acc_lin_tr:.4f}  ({acc_lin_tr*100:.1f}%)')
print(f'  Test  accuracy : {acc_lin_te:.4f}  ({acc_lin_te*100:.1f}%)')
print(f'  Gap (train-test): {(acc_lin_tr-acc_lin_te)*100:.1f} pp')
print()
print(f'  Number of support vectors : {svm_lin.n_support_}  '
      f'(class 0: {svm_lin.n_support_[0]}, class 1: {svm_lin.n_support_[1]})')
print()
print('Confusion Matrix (test set):')
print(f'  TN = {TN}  FP = {FP}')
print(f'  FN = {FN}  TP = {TP}')
print()
print('Classification Report (test set):')
print(classification_report(y_te, y_pred_lin_te,
                             target_names=['No Click (0)', 'Clicked (1)']))

---

## Step 2.4 — RBF SVM and Kernel Comparison

The **Radial Basis Function (RBF) kernel** implicitly projects the data into an infinite-dimensional space where a linear hyperplane can separate classes that are non-linearly entangled in the original space:

$$K(x_i, x_j) = \exp\left(-\gamma \|x_i - x_j\|^2\right)$$

The parameter **γ (gamma)** controls the "reach" of each support vector:
- **Small γ** → each point influences a wide region → smoother, simpler boundary
- **Large γ** → each point influences only nearby points → complex, possibly over-fitted boundary

**Session 31 connection:** The kernel trick is the key innovation that made SVMs practical for real-world non-linear problems. The data does not need to be explicitly mapped — the kernel function computes inner products in the higher-dimensional space efficiently.

In [ ]:
# ── Step 2.4 ─ RBF SVM and Kernel Comparison ─────────────────────────────────

svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=10)
svm_rbf.fit(X_tr_sc, y_tr)

y_pred_rbf_tr = svm_rbf.predict(X_tr_sc)
y_pred_rbf_te = svm_rbf.predict(X_te_sc)

acc_rbf_tr = accuracy_score(y_tr, y_pred_rbf_tr)
acc_rbf_te = accuracy_score(y_te, y_pred_rbf_te)

cm_rbf = confusion_matrix(y_te, y_pred_rbf_te)
TN_r, FP_r, FN_r, TP_r = cm_rbf.ravel()

# Also try polynomial kernel for completeness
svm_poly = SVC(kernel='poly', degree=3, C=1.0, random_state=10)
svm_poly.fit(X_tr_sc, y_tr)
acc_poly_te = accuracy_score(y_te, svm_poly.predict(X_te_sc))
acc_poly_tr = accuracy_score(y_tr, svm_poly.predict(X_tr_sc))

print('=== Step 2.4 — Kernel Comparison ===')
print()
print(f'{"Kernel":<12} {"Train Acc":>10} {"Test Acc":>10} {"Gap (pp)":>10}')
print('-' * 48)
for kernel, tr_acc, te_acc in [
        ('Linear',     acc_lin_tr,  acc_lin_te),
        ('RBF',        acc_rbf_tr,  acc_rbf_te),
        ('Polynomial', acc_poly_tr, acc_poly_te),
]:
    gap = (tr_acc - te_acc) * 100
    print(f'{kernel:<12} {tr_acc*100:>9.1f}%  {te_acc*100:>9.1f}%  {gap:>9.1f} pp')

print()
print(f'RBF support vectors : {svm_rbf.n_support_}  '
      f'(class 0: {svm_rbf.n_support_[0]}, class 1: {svm_rbf.n_support_[1]})')
print()

# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cell_lbls = [['TN','FP'],['FN','TP']]

for ax, cm_, title in zip(axes,
                           [cm_lin, cm_rbf],
                           ['Linear SVM', 'RBF SVM']):
    im = ax.imshow(cm_, cmap='Blues', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    thresh = cm_.max() / 2.0
    for i in range(2):
        for j in range(2):
            color = 'white' if cm_[i,j] > thresh else 'black'
            ax.text(j, i, f'{cell_lbls[i][j]}\n{cm_[i,j]}',
                    ha='center', va='center', fontsize=13,
                    fontweight='bold', color=color)
    ax.set_xticks([0,1]);  ax.set_yticks([0,1])
    ax.set_xticklabels(['Pred: No Click','Pred: Click'], fontsize=9)
    ax.set_yticklabels(['Actual: No Click','Actual: Click'], fontsize=9)
    acc_te_ = accuracy_score(y_te, svm_lin.predict(X_te_sc) if 'Lin' in title
                             else svm_rbf.predict(X_te_sc))
    ax.set_title(f'Step 2.4 — {title}\nTest Acc = {acc_te_*100:.1f}%', fontsize=11)

plt.tight_layout()
plt.show()

---

## Step 2.5 — Hyperparameter Tuning with Grid Search

The performance of an SVM depends heavily on **C** and **γ**. Instead of guessing, we use **Grid Search Cross-Validation** to evaluate every combination on a grid of candidate values:

| Hyperparameter | Candidate Values | Effect |
|----------------|-----------------|--------|
| `C` | 0.01, 0.1, 1, 10, 100 | Regularization strength |
| `gamma` | 0.001, 0.01, 0.1, 1, 'scale' | RBF kernel bandwidth |

We wrap the scaler and SVM in a `Pipeline` to prevent data leakage during cross-validation (same reason as Activity 9's cross-validation step).

**Session 31 connection:** Grid Search is the brute-force approach to hyperparameter selection. For larger grids, Randomized Search or Bayesian Optimization are more efficient — but Grid Search is ideal for learning because every combination is visible and interpretable.

In [ ]:
# ── Step 2.5 ─ Grid Search Cross-Validation ────────────────────────────────────

pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', random_state=10))
])

param_grid = {
    'svm__C'     : [0.01, 0.1, 1, 10, 100],
    'svm__gamma' : [0.001, 0.01, 0.1, 1, 'scale'],
}

grid_search = GridSearchCV(
    pipe_svm,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_tr, y_tr)   # raw data — pipeline handles scaling internally

best_params = grid_search.best_params_
best_cv_acc = grid_search.best_score_

print('=== Step 2.5 — Grid Search Results ===')
print(f'  Grid size         : {len(grid_search.cv_results_["params"])} combinations (5 C × 5 gamma)')
print(f'  Cross-validation  : 5-fold')
print(f'  Best C            : {best_params["svm__C"]}')
print(f'  Best gamma        : {best_params["svm__gamma"]}')
print(f'  Best CV accuracy  : {best_cv_acc*100:.2f}%')
print()

# Heatmap of mean CV accuracy across C x gamma grid
results   = grid_search.cv_results_
C_vals    = [0.01, 0.1, 1, 10, 100]
gamma_vals= [0.001, 0.01, 0.1, 1, 'scale']

# Build score matrix (rows = C, cols = gamma)
score_matrix = np.zeros((len(C_vals), len(gamma_vals)))
for i, c_val in enumerate(C_vals):
    for j, g_val in enumerate(gamma_vals):
        mask = [(p['svm__C'] == c_val and p['svm__gamma'] == g_val)
                for p in results['params']]
        score_matrix[i, j] = results['mean_test_score'][mask].mean()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(score_matrix, cmap='YlGnBu', vmin=score_matrix.min()-0.01,
               vmax=score_matrix.max()+0.005, interpolation='nearest')
plt.colorbar(im, ax=ax, label='Mean CV Accuracy')
ax.set_xticks(range(len(gamma_vals)))
ax.set_yticks(range(len(C_vals)))
ax.set_xticklabels([str(g) for g in gamma_vals], fontsize=10)
ax.set_yticklabels([str(c) for c in C_vals], fontsize=10)
ax.set_xlabel('gamma', fontsize=11)
ax.set_ylabel('C', fontsize=11)
ax.set_title(f'Step 2.5 — Grid Search CV Accuracy Heatmap\n'
             f'Best: C={best_params["svm__C"]}, gamma={best_params["svm__gamma"]}, '
             f'acc={best_cv_acc*100:.2f}%', fontsize=12)
for i in range(len(C_vals)):
    for j in range(len(gamma_vals)):
        ax.text(j, i, f'{score_matrix[i,j]*100:.1f}%',
                ha='center', va='center', fontsize=9,
                color='white' if score_matrix[i,j] > score_matrix.mean() else 'black')
plt.tight_layout()
plt.show()

---

## Step 2.6 — Best Model Evaluation

We retrain the SVM with the best hyperparameters found by Grid Search and evaluate it fully on the held-out test set.

In [ ]:
# ── Step 2.6 ─ Best Model Evaluation ─────────────────────────────────────────

# Best estimator from grid search (already fitted)
best_svm = grid_search.best_estimator_

y_pred_best = best_svm.predict(X_te)
acc_best    = accuracy_score(y_te, y_pred_best)

cm_best = confusion_matrix(y_te, y_pred_best)
TN_b, FP_b, FN_b, TP_b = cm_best.ravel()

print('=== Step 2.6 — Best SVM Evaluation ===')
print(f'  Best params   : C={best_params["svm__C"]}, gamma={best_params["svm__gamma"]}')
print(f'  Test accuracy : {acc_best:.4f}  ({acc_best*100:.1f}%)')
print()
print('Confusion Matrix (test set):')
print(f'  TN = {TN_b:3d}  FP = {FP_b:3d}')
print(f'  FN = {FN_b:3d}  TP = {TP_b:3d}')
print()
print('Classification Report:')
report = classification_report(y_te, y_pred_best,
                                target_names=['No Click (0)', 'Clicked (1)'])
print(report)

# Manual metric verification
prec_best   = TP_b / (TP_b + FP_b) if (TP_b + FP_b) > 0 else 0
recall_best = TP_b / (TP_b + FN_b) if (TP_b + FN_b) > 0 else 0
f1_best     = 2 * prec_best * recall_best / (prec_best + recall_best) \
              if (prec_best + recall_best) > 0 else 0

print('Manual verification for Clicked class (1):')
print(f'  Precision = TP/(TP+FP) = {TP_b}/({TP_b}+{FP_b}) = {prec_best:.4f}')
print(f'  Recall    = TP/(TP+FN) = {TP_b}/({TP_b}+{FN_b}) = {recall_best:.4f}')
print(f'  F1-score  = 2*P*R/(P+R)              = {f1_best:.4f}')

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_best, cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
thresh = cm_best.max() / 2.0
cell_lbls = [['TN','FP'],['FN','TP']]
for i in range(2):
    for j in range(2):
        color = 'white' if cm_best[i,j] > thresh else 'black'
        ax.text(j, i, f'{cell_lbls[i][j]}\n{cm_best[i,j]}',
                ha='center', va='center', fontsize=14, fontweight='bold', color=color)
ax.set_xticks([0,1]);  ax.set_yticks([0,1])
ax.set_xticklabels(['Predicted: No Click','Predicted: Click'], fontsize=10)
ax.set_yticklabels(['Actual: No Click','Actual: Click'], fontsize=10)
ax.set_title(f'Step 2.6 — Best SVM Confusion Matrix\nAccuracy = {acc_best*100:.1f}%',
             fontsize=12)
plt.tight_layout()
plt.show()

---

## Step 2.7 — Decision Boundary Visualization

To visualize the SVM decision boundary, we train a 2D model using only `income_k` and `pages_viewed` — the two features most correlated with the click outcome. We compare the **Linear** and **RBF** boundaries side by side.

**What to look for:**
- The **Linear** boundary is a straight line — it separates the two classes with a single hyperplane.
- The **RBF** boundary is a curved surface — it can wrap around clusters of one class.
- The **shaded margin band** shows the zone between the support vectors. A wider margin = more confident separation.
- **Support vectors** are circled — notice how few points actually define the boundary.

In [ ]:
# ── Step 2.7 ─ Decision Boundary (2D, income_k vs pages_viewed) ───────────────

feat_2d   = ['income_k', 'pages_viewed']
idx_2d    = [features_svm.index(f) for f in feat_2d]

X_2d      = X_svm[:, idx_2d]
scaler_2d = StandardScaler()

X_2d_tr, X_2d_te, y_2d_tr, y_2d_te = train_test_split(
    X_2d, y_svm, test_size=0.20, random_state=10, stratify=y_svm
)

X_2d_tr_sc = scaler_2d.fit_transform(X_2d_tr)
X_2d_te_sc = scaler_2d.transform(X_2d_te)

configs = [
    ('Linear SVM', SVC(kernel='linear', C=1.0, random_state=10)),
    ('RBF SVM',    SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=10)),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, clf) in zip(axes, configs):
    clf.fit(X_2d_tr_sc, y_2d_tr)
    te_acc_2d = accuracy_score(y_2d_te, clf.predict(X_2d_te_sc))

    # Build meshgrid in scaled space
    x_min, x_max = X_2d_tr_sc[:,0].min()-0.5, X_2d_tr_sc[:,0].max()+0.5
    y_min, y_max = X_2d_tr_sc[:,1].min()-0.5, X_2d_tr_sc[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.20, cmap='coolwarm')
    ax.contour( xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)

    # Decision function for margin
    if hasattr(clf, 'decision_function'):
        Zdf = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contour(xx, yy, Zdf, levels=[-1, 1],
                   linestyles='--', colors='grey', linewidths=1)

    # Training points
    for cls, color, marker, label in zip([0,1],
                                          ['steelblue','firebrick'],
                                          ['o','s'],
                                          ['No Click','Clicked']):
        mask = y_2d_tr == cls
        ax.scatter(X_2d_tr_sc[mask,0], X_2d_tr_sc[mask,1],
                   c=color, marker=marker, s=30, alpha=0.6,
                   edgecolors='white', linewidths=0.3, label=label)

    # Support vectors
    ax.scatter(clf.support_vectors_[:,0], clf.support_vectors_[:,1],
               s=120, facecolors='none', edgecolors='black', linewidths=1.2,
               label=f'Support vectors (n={len(clf.support_vectors_)})', zorder=5)

    ax.set_xlabel('income_k (scaled)', fontsize=10)
    ax.set_ylabel('pages_viewed (scaled)', fontsize=10)
    ax.set_title(f'Step 2.7 — {title}\nTest Acc = {te_acc_2d*100:.1f}%  '
                 f'(2-feature model)', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.2)

plt.suptitle('Decision Boundary: income_k vs pages_viewed', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

for title, clf in configs:
    clf.fit(X_2d_tr_sc, y_2d_tr)
    print(f'{title:20s} support vectors: {clf.n_support_}  '
          f'(class 0: {clf.n_support_[0]}, class 1: {clf.n_support_[1]})')

---

## Step 2.8 — Interactive Widget: Explore C, Kernel, and Decision Boundary

Use the controls below to experiment with SVM hyperparameters in real time.

**Experiments to try:**
1. Keep `kernel='rbf'` and increase C from 0.1 → 100. How does the decision boundary change? Does the number of support vectors increase or decrease?
2. Switch to `kernel='linear'`. How does C affect the width of the margin band (dashed lines)?
3. Set `kernel='rbf'`, `C=100` — do you see signs of overfitting (very jagged boundary)? How does this compare to C=0.1?
4. Compare the train/test accuracy gap at C=0.01 vs C=100 for the RBF kernel.

**Reflection Question (answer in the cell below the widget):**  
At what value of C does the RBF SVM begin to overfit on the training data (train accuracy much higher than test accuracy)? What does this tell you about the role of regularization in SVMs?

In [ ]:
# ── Step 2.8 ─ Interactive SVM Widget ────────────────────────────────────────

def run_svm_widget(kernel, C):
    clf_w = SVC(kernel=kernel, C=C, gamma='scale', random_state=10)
    clf_w.fit(X_2d_tr_sc, y_2d_tr)

    acc_tr_w = accuracy_score(y_2d_tr, clf_w.predict(X_2d_tr_sc))
    acc_te_w = accuracy_score(y_2d_te, clf_w.predict(X_2d_te_sc))
    gap_w    = acc_tr_w - acc_te_w

    # Decision boundary
    x_min, x_max = X_2d_tr_sc[:,0].min()-0.5, X_2d_tr_sc[:,0].max()+0.5
    y_min, y_max = X_2d_tr_sc[:,1].min()-0.5, X_2d_tr_sc[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                          np.linspace(y_min, y_max, 250))
    Z = clf_w.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Kernel={kernel}  |  C={C}  |  '
        f'Train Acc={acc_tr_w*100:.1f}%  |  Test Acc={acc_te_w*100:.1f}%  |  '
        f'Gap={gap_w*100:.1f} pp' +
        ('  <<< WARNING: overfitting' if gap_w > 0.10 else ''),
        fontsize=11
    )

    # Decision boundary plot
    axes[0].contourf(xx, yy, Z, alpha=0.20, cmap='coolwarm')
    axes[0].contour( xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)
    if hasattr(clf_w, 'decision_function'):
        Zdf = clf_w.decision_function(np.c_[xx.ravel(),
                                             yy.ravel()]).reshape(xx.shape)
        axes[0].contour(xx, yy, Zdf, levels=[-1,1],
                        linestyles='--', colors='grey', linewidths=1)
    for cls, col, mrk in zip([0,1],['steelblue','firebrick'],['o','s']):
        mask = y_2d_tr == cls
        axes[0].scatter(X_2d_tr_sc[mask,0], X_2d_tr_sc[mask,1],
                        c=col, marker=mrk, s=30, alpha=0.6,
                        edgecolors='white', linewidths=0.3)
    axes[0].scatter(clf_w.support_vectors_[:,0], clf_w.support_vectors_[:,1],
                    s=110, facecolors='none', edgecolors='black', linewidths=1.2, zorder=5)
    axes[0].set_xlabel('income_k (scaled)', fontsize=10)
    axes[0].set_ylabel('pages_viewed (scaled)', fontsize=10)
    axes[0].set_title(f'Decision Boundary\n'
                      f'Support vectors: {len(clf_w.support_vectors_)}', fontsize=11)
    axes[0].grid(True, alpha=0.2)

    # Accuracy comparison bar
    bars_w = axes[1].bar(['Train Accuracy', 'Test Accuracy'],
                          [acc_tr_w*100, acc_te_w*100],
                          color=['steelblue','firebrick'], edgecolor='white', width=0.4)
    for bar_w, val in zip(bars_w, [acc_tr_w*100, acc_te_w*100]):
        axes[1].text(bar_w.get_x() + bar_w.get_width()/2,
                     bar_w.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=12,
                     fontweight='bold')
    axes[1].set_ylim(40, 105)
    axes[1].set_ylabel('Accuracy (%)', fontsize=11)
    axes[1].set_title('Train vs Test Accuracy', fontsize=11)
    axes[1].axhline(50, color='grey', linestyle=':', linewidth=1, label='Random baseline')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()


widgets.interact(
    run_svm_widget,
    kernel=widgets.Dropdown(
        options=['linear', 'rbf', 'poly'],
        value='rbf',
        description='Kernel:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='40%')
    ),
    C=widgets.SelectionSlider(
        options=[0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
        value=1,
        description='C (regularization):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )
)

**Reflection Question Q2 — answer here:**

*Write your answer in this Markdown cell.*

---

## Conclusions

This notebook built a complete AI-driven digital marketing analytics pipeline for NovaPulse Media / StyleHub:

| | Task 1 — K-Means | Task 2 — SVM |
|--|--|--|
| **Type** | Unsupervised (no labels) | Supervised (labeled history) |
| **Goal** | Discover behavioral segments | Predict ad clicks |
| **Key concept** | Centroid, inertia, elbow method | Hyperplane, margin, kernel trick |
| **Tuning** | Choose K via elbow + silhouette | Choose C and γ via Grid Search CV |
| **Output** | 3 customer personas | Binary click prediction |
| **Session** | 29 | 31 |

**Key takeaways:**
- Feature normalization is critical for both K-Means (distance-based) and SVM (margin-based)
- The elbow method provides a principled way to choose K — silhouette score provides a complementary validation
- SVMs with RBF kernels can capture non-linear decision boundaries that linear models miss
- Grid Search CV finds the best C/γ combination without manual tuning — but always evaluate on a held-out test set, not just CV score
- In marketing, model outputs must be interpreted in business terms: cluster labels become personas; click probabilities become ad targeting priorities

---

## Reflection Questions

*Write your answers in the Markdown cells below each question.*

**Q1.** In Step 1.3, we apply `StandardScaler` to the K-Means data and fit it on the **entire dataset** (not just a training split). Explain why this is acceptable for unsupervised clustering, whereas in Activity 9 (supervised learning) we had to fit the scaler only on the training set.

*Answer: ...*

---

**Q2.** The Elbow Method (Step 1.4) shows a noticeable bend at K = 3. However, the elbow is subjective — two analysts might choose K = 3 or K = 4. Describe one additional technique (beyond the elbow and silhouette score) that could help confirm the right number of clusters in a real business context.

*Answer: ...*

---

**Q3.** In Step 2.4, the RBF SVM and Linear SVM are compared. Describe a real-world digital marketing dataset where you would expect the **Linear SVM to outperform** the RBF SVM, and explain why.

*Answer: ...*

---

**Q4.** The Grid Search in Step 2.5 evaluates 25 combinations (5 C × 5 gamma) each with 5-fold CV — totaling 125 model fits. If StyleHub's full dataset had 10 million customers and 50 features, what problems would this approach face? Name two more efficient alternatives discussed in Session 31.

*Answer: ...*

---

**Q5.** The K-Means cluster labels (0, 1, 2) from Task 1 could be used as a new feature in the SVM model of Task 2 — adding a "customer segment" input to the ad click predictor. What would be the potential benefit of this combined approach? What risk would you need to watch for?

*Answer: ...*